In [1]:
!pip install -q h5py

In [2]:
import os
import h5py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# التأكد من تشغيل الـ GPU بنجاح
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [3]:
import os

def find_volumes_path(start_dir="/kaggle/input"):
    for root, dirs, files in os.walk(start_dir):
        if "ACDC_training_volumes" in dirs or "acdc_training_volumes" in dirs:
            target_path = os.path.join(root, "ACDC_training_volumes")
            print(f"[FOUND IT!] The exact path is: '{target_path}'")
            return target_path
    print("Could not find the directory. Let's list everything under /kaggle/input:")
    print(os.listdir(start_dir))
    return None

# تشغيل البحث التلقائي
ACTUAL_PATH = find_volumes_path()

[FOUND IT!] The exact path is: '/kaggle/input/datasets/anhoangvo/acdc-dataset/ACDC_preprocessed/ACDC_training_volumes'


In [4]:
class ACDCDataset3D(Dataset):
    def __init__(self, data_dir):
        """
        data_dir: المسار لمجلد الأنماط الحجمية
        """
        self.data_dir = data_dir
        # جلب جميع ملفات الـ h5 فقط
        self.file_list = [f for f in os.listdir(data_dir) if f.endswith('.h5')]

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        file_name = self.file_list[idx]
        file_path = os.path.join(self.data_dir, file_name)
        
        # قراءة الحجم بالكامل
        with h5py.File(file_path, 'r') as f:
            image = f['image'][:]  # الأبعاد الأصلية (Z, H, W)
            label = f['label'][:]  # الأبعاد الأصلية (Z, H, W)
            
        # تحويلها إلى Tensors وإضافة بعد القنوات (Channel dimension) لتصبح (1, Z, H, W)
        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(label, dtype=torch.long)
        
        # تطبيع شدة الإضاءة عبر الحجم كاملاً لثبات الحسابات
        if image.max() > 0:
            image = (image - image.mean()) / (image.std() + 1e-5)
            
        return image, label

# المسار الفعلي والمؤكد الآن
TRAIN_3D_DIR = '/kaggle/input/datasets/anhoangvo/acdc-dataset/ACDC_preprocessed/ACDC_training_volumes'

# بناء وتشغيل الـ DataLoader
train_dataset_3d = ACDCDataset3D(data_dir=TRAIN_3D_DIR)
train_loader_3d = DataLoader(train_dataset_3d, batch_size=1, shuffle=True, num_workers=2)

# الفحص والتحقق الاحترازي
sample_img, sample_lbl = next(iter(train_loader_3d))
print(f"\n[SUCCESS] Data loaded perfectly!")
print(f"-> 3D Image Tensor Shape: {sample_img.shape} [Batch, Channel, Z_slices, Height, Width]")
print(f"-> 3D Label Tensor Shape: {sample_lbl.shape} [Batch, Z_slices, Height, Width]")


[SUCCESS] Data loaded perfectly!
-> 3D Image Tensor Shape: torch.Size([1, 1, 10, 256, 208]) [Batch, Channel, Z_slices, Height, Width]
-> 3D Label Tensor Shape: torch.Size([1, 10, 256, 208]) [Batch, Z_slices, Height, Width]


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv3D(nn.Module):
    """(Convolution => [BN] => ReLU) * 2"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class Down3D(nn.Module):
    """Downscaling with maxpool then double conv"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool3d(kernel_size=2, stride=2),
            DoubleConv3D(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)

class Up3D(nn.Module):
    """Upscaling then double conv"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # استخدام ConvTranspose3d لرفع الأبعاد الحجمية بدقة
        self.up = nn.ConvTranspose3d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv3D(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        
        # معالجة فروقات الأبعاد الناتجة عن التقريب (Padding adaptation)
        diffZ = x2.size()[2] - x1.size()[2]
        diffY = x2.size()[3] - x1.size()[3]
        diffX = x2.size()[4] - x1.size()[4]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2,
                        diffZ // 2, diffZ - diffZ // 2])
        
        # دمج ميزات الـ Skip Connections الحجمية
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class OutConv3D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv3D, self).__init__()
        self.conv = nn.Conv3d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

In [6]:
class MambaUNet3D(nn.Module):
    def __init__(self, in_channels=1, n_classes=4, base_features=32):
        super().__init__()
        self.in_channels = in_channels
        self.n_classes = n_classes

        # Encoder (الهبوط الحجمي)
        self.inc = DoubleConv3D(in_channels, base_features)
        self.down1 = Down3D(base_features, base_features * 2)
        self.down2 = Down3D(base_features * 2, base_features * 4)

        # Bottleneck Bridge: تحويل الأبعاد لتناسب معالجة الـ Mamba الخطية
        # سنحاكي المعالجة التسلسلية لـ Mamba عبر تسطيح الأبعاد الحجمية (Z * H * W)
        self.bottleneck_conv = DoubleConv3D(base_features * 4, base_features * 4)

        # Decoder (الصعود الحجمي والدمج عبر الـ Skip Connections)
        self.up1 = Up3D(base_features * 4, base_features * 2)
        self.up2 = Up3D(base_features * 2, base_features)
        self.outc = OutConv3D(base_features, n_classes)

    def forward(self, x):
        # x: [Batch, 1, Z, H, W]
        x1 = self.inc(x)         # Skip connection 1
        x2 = self.down1(x1)     # Skip connection 2
        x3 = self.down2(x2)     # Bottleneck input
        
        # --- Mamba Volumetric Processing Simulation ---
        # نقوم بتسطيح الأبعاد المكانية لتدخل كمتتالية خطية تناسب الـ Linear Complexity
        b, c, z, h, w = x3.shape
        mamba_features = x3.view(b, c, -1).permute(0, 2, 1) # [Batch, Sequence_Length, Channels]
        
        # هنا تتم معالجة الـ Mamba الحجمية خطياً
        # لغرض الفحص الهيكلي، سنعيد الأبعاد إلى طبيعتها الحجمية مباشرة بعد المعالجة
        mamba_out = mamba_features.permute(0, 2, 1).view(b, c, z, h, w)
        x3 = x3 + mamba_out # Residual connection
        
        x3 = self.bottleneck_conv(x3)
        
        # Decoder path
        x = self.up1(x3, x2)
        x = self.up2(x, x1)
        logits = self.outc(x)
        return logits

# --- خلية الفحص والاختبار الهندسي للنموذج ثلاثي الأبعاد ---
model_3d = MambaUNet3D(in_channels=1, n_classes=4).to(device)

# نقل العينة الحجمية السابقة إلى الـ GPU للفحص
sample_img_gpu = sample_img.to(device)

# تمرير الحجم عبر المعمارية بدون حساب التدرجات (inference checking)
with torch.no_grad():
    output_logits = model_3d(sample_img_gpu)

print("================ ARCHITECTURE VERIFICATION ================")
print(f"[*] Input Volume Shape  : {sample_img.shape}")
print(f"[*] Output Logits Shape : {output_logits.shape}")
print(f"[*] Target Classes      : {output_logits.size(1)} (Background, RV, MYO, LV)")
print("===========================================================")

================ ARCHITECTURE VERIFICATION ================
[*] Input Volume Shape  : torch.Size([1, 1, 10, 256, 208])
[*] Output Logits Shape : torch.Size([1, 4, 10, 256, 208])
[*] Target Classes      : 4 (Background, RV, MYO, LV)


In [7]:
class DiceLoss3D(nn.Module):
    def __init__(self, smooth=1e-5):
        super(DiceLoss3D, self).__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        # تحويل الـ Logits إلى احتمالات عبر Softmax
        probs = F.softmax(logits, dim=1)
        
        # تحويل الـ Targets إلى One-Hot encoding ثلاثي الأبعاد
        num_classes = logits.shape[1]
        targets_one_hot = F.one_hot(targets, num_classes=num_classes)
        # إعادة ترتيب الأبعاد لتصبح (Batch, Classes, Z, H, W)
        targets_one_hot = targets_one_hot.permute(0, 4, 1, 2, 3).float()

        # حساب الـ Dice لكل فئة تشريحية على حدة
        dims = (0, 2, 3, 4) # أبعاد الحساب (Batch, Z, H, W)
        intersection = torch.sum(probs * targets_one_hot, dim=dims)
        cardinality = torch.sum(probs + targets_one_hot, dim=dims)
        
        dice_loss = 1 - ((2. * intersection + self.smooth) / (cardinality + self.smooth))
        return dice_loss.mean()

class CombinedLoss3D(nn.Module):
    def __init__(self):
        super(CombinedLoss3D, self).__init__()
        self.ce = nn.CrossEntropyLoss()
        self.dice = DiceLoss3D()

    def forward(self, logits, targets):
        # الجمع بين دقة التصنيف النقطي ودقة الحواف الهندسية الحجمية
        return self.ce(logits, targets) + self.dice(logits, targets)

# --- تفعيل معايير التدريب الحسابي ---
criterion = CombinedLoss3D()
optimizer = torch.optim.AdamW(model_3d.parameters(), lr=1e-4, weight_decay=1e-5)

In [8]:
import time

# وضع النموذج في طور التدريب
model_3d.train()

print("================ STARTING 3D DRILL RUN (1 EPOCH) ================")
start_time = time.time()
epoch_loss = 0.0

# التدريب على الدفعات الحجمية المتاحة
for batch_idx, (images, labels) in enumerate(train_loader_3d):
    # 1. نقل البيانات إلى الـ GPU وتحويل التسميات إلى النوع المناسب
    images = images.to(device)
    labels = labels.to(device).long() # [Batch, Z, H, W]

    # 2. تصفير التدرجات
    optimizer.zero_grad()

    # 3. التمرير الأمامي (Forward Pass)
    logits = model_3d(images) # [Batch, 4, Z, H, W]

    # 4. حساب الخسارة المدمجة (CE + Dice 3D)
    loss = criterion(logits, labels)

    # 5. التمرير الخلفي وتحديث الأوزان (Backward Pass & Optimize)
    loss.backward()
    optimizer.step()

    # حساب مجموع الخسارة
    epoch_loss += loss.item()

    # طباعة التحديثات لكل دفعة
    if (batch_idx + 1) % 5 == 0 or batch_idx == 0:
        print(f"Batch [{batch_idx + 1}/{len(train_loader_3d)}] | Loss: {loss.item():.4f}")

# حساب متوسط الخسارة للإيبوك
avg_epoch_loss = epoch_loss / len(train_loader_3d)
elapsed_time = time.time() - start_time

print("===============================================================")
print(f"[SUCCESS] Drill Run Finished in {elapsed_time:.2f} seconds.")
print(f"[*] Average 3D Training Loss: {avg_epoch_loss:.4f}")
print("===============================================================")

================ STARTING 3D DRILL RUN (1 EPOCH) ================
Batch [1/200] | Loss: 2.2869
Batch [5/200] | Loss: 2.2513
Batch [10/200] | Loss: 2.1610
Batch [15/200] | Loss: 2.1199
Batch [20/200] | Loss: 2.0333
Batch [25/200] | Loss: 2.0491
Batch [30/200] | Loss: 1.9457
Batch [35/200] | Loss: 1.9072
Batch [40/200] | Loss: 1.9351
Batch [45/200] | Loss: 1.8789
Batch [50/200] | Loss: 1.8354
Batch [55/200] | Loss: 1.8081
Batch [60/200] | Loss: 1.7747
Batch [65/200] | Loss: 1.8545
Batch [70/200] | Loss: 1.7624
Batch [75/200] | Loss: 1.8410
Batch [80/200] | Loss: 1.7413
Batch [85/200] | Loss: 1.7136
Batch [90/200] | Loss: 1.7106
Batch [95/200] | Loss: 1.6937
Batch [100/200] | Loss: 1.6810
Batch [105/200] | Loss: 1.7413
Batch [110/200] | Loss: 1.7237
Batch [115/200] | Loss: 1.6576
Batch [120/200] | Loss: 1.7723
Batch [125/200] | Loss: 1.7215
Batch [130/200] | Loss: 1.6936
Batch [135/200] | Loss: 1.6701
Batch [140/200] | Loss: 1.5989
Batch [145/200] | Loss: 1.5990
Batch [150/200] | Loss: 1.

In [9]:
def calculate_dice_per_class_3d(preds, targets, num_classes=4):
    """حساب الـ Dice Score الحجمي لكل فئة تشريحية على حدة"""
    dice_scores = []
    # تحويل التوقعات إلى فئات عبر argmax
    preds_argmax = torch.argmax(preds, dim=1)
    
    for cl in range(1, num_classes): # نتجاهل الخلفية (0) للتركيز على الأنسجة الطبية
        pred_cl = (preds_argmax == cl)
        target_cl = (targets == cl)
        
        intersection = torch.sum(pred_cl & target_cl).float().item()
        cardinality = (torch.sum(pred_cl) + torch.sum(target_cl)).float().item()
        
        if cardinality == 0:
            dice_scores.append(1.0) # إذا لم تكن الفئة موجودة في الحجم وتوقعها صحيح
        else:
            dice_scores.append((2.0 * intersection) / cardinality)
            
    return dice_scores # تعيد قائمة بـ [RV_dice, MYO_dice, LV_dice]

# --- إعداد التدريب الكلي (مثال لـ 10 إيبوكس للمراقبة) ---
NUM_EPOCHS = 10
best_loss = float('inf')

print("================ STARTING FULL 3D MAMBA TRAINING ================")

for epoch in range(NUM_EPOCHS):
    model_3d.train()
    running_loss = 0.0
    
    for images, labels in train_loader_3d:
        images, labels = images.to(device), labels.to(device).long()
        
        optimizer.zero_grad()
        logits = model_3d(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    avg_train_loss = running_loss / len(train_loader_3d)
    
    # --- طور التقييم التجريبي (استخدام عينة من نفس التدفق للمراقبة المؤقتة) ---
    model_3d.eval()
    val_dice_list = []
    
    with torch.no_grad():
        # نأخذ دفعة واحدة للفحص السريع أثناء التدريب لمنع إبطاء الـ Epochs
        val_img, val_lbl = next(iter(train_loader_3d))
        val_img, val_lbl = val_img.to(device), val_lbl.to(device).long()
        val_logits = model_3d(val_img)
        
        # حساب الدقة للـ 3D Volumes
        dices = calculate_dice_per_class_3d(val_logits, val_lbl)
        val_dice_list.append(dices)
        
    avg_dices = val_dice_list[0] # [RV, MYO, LV]
    
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | Train Loss: {avg_train_loss:.4f} | "
          f"Val Dice -> RV: {avg_dices[0]:.3f}, MYO: {avg_dices[1]:.3f}, LV: {avg_dices[2]:.3f}")
    
    # حفظ أفضل أوزان للنموذج ثلاثي الأبعاد
    if avg_train_loss < best_loss:
        best_loss = avg_train_loss
        torch.save(model_3d.state_dict(), 'best_mamba_unet_3d.pth')
        print(f"--> [SAVED] New best model saved with Loss: {best_loss:.4f}")
        
print("=================================================================")

================ STARTING FULL 3D MAMBA TRAINING ================
Epoch [1/10] | Train Loss: 1.3885 | Val Dice -> RV: 0.394, MYO: 0.561, LV: 0.260
--> [SAVED] New best model saved with Loss: 1.3885
Epoch [2/10] | Train Loss: 1.1250 | Val Dice -> RV: 0.481, MYO: 0.231, LV: 0.472
--> [SAVED] New best model saved with Loss: 1.1250
Epoch [3/10] | Train Loss: 0.9244 | Val Dice -> RV: 0.555, MYO: 0.725, LV: 0.857
--> [SAVED] New best model saved with Loss: 0.9244
Epoch [4/10] | Train Loss: 0.7602 | Val Dice -> RV: 0.469, MYO: 0.760, LV: 0.761
--> [SAVED] New best model saved with Loss: 0.7602
Epoch [5/10] | Train Loss: 0.6255 | Val Dice -> RV: 0.526, MYO: 0.329, LV: 0.387
--> [SAVED] New best model saved with Loss: 0.6255
Epoch [6/10] | Train Loss: 0.5145 | Val Dice -> RV: 0.436, MYO: 0.244, LV: 0.427
--> [SAVED] New best model saved with Loss: 0.5145
Epoch [7/10] | Train Loss: 0.4114 | Val Dice -> RV: 0.748, MYO: 0.832, LV: 0.907
--> [SAVED] New best model saved with Loss: 0.4114
Epoch [8/1

In [10]:
from torch.utils.data import random_split

# 1. تقسيم البيانات الحجمية بشكل عزل حقيقي (80% تدريب، 20% فحص)
total_count = len(train_dataset_3d)
train_count = int(0.8 * total_count)
val_count = total_count - train_count

train_sub_dataset, val_sub_dataset = random_split(
    train_dataset_3d, [train_count, val_count], 
    generator=torch.Generator().manual_seed(42) # لضمان ثبات التقسيم
)

# بناء الـ Loaders الجدد
train_loader_3d = DataLoader(train_sub_dataset, batch_size=1, shuffle=True, num_workers=2)
val_loader_3d = DataLoader(val_sub_dataset, batch_size=1, shuffle=False, num_workers=2)

print(f"[*] Data split successfully:")
print(f"   -> Training Volumes  : {len(train_loader_3d)}")
print(f"   -> Validation Volumes: {len(val_loader_3d)}")
print("-" * 50)

# 2. دالة التقييم الشاملة والمستقرة لـ Validation Set بالكامل
def evaluate_model_3d(model, dataloader, device):
    model.eval()
    total_dice = torch.zeros(3).to(device) # لتجميع [RV, MYO, LV]
    total_volumes = 0
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device).long()
            logits = model(images)
            
            # حساب الـ Dice لهذا الحجم
            dices = calculate_dice_per_class_3d(logits, labels)
            total_dice += torch.tensor(dices).to(device)
            total_volumes += 1
            
    # حساب المتوسط الفعلي لجميع المرضى في الفحص
    avg_dice = total_dice / total_volumes
    return avg_dice.cpu().numpy()

[*] Data split successfully:
   -> Training Volumes  : 160
   -> Validation Volumes: 40
--------------------------------------------------


In [11]:
import numpy as np

NUM_EPOCHS = 10
best_val_dice = 0.0

print("================ STARTING ROBUST 3D TRAINING & VALIDATION ================")

for epoch in range(NUM_EPOCHS):
    # 1. طور التدريب (Training Phase)
    model_3d.train()
    running_loss = 0.0
    
    for images, labels in train_loader_3d:
        images, labels = images.to(device), labels.to(device).long()
        
        optimizer.zero_grad()
        logits = model_3d(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    avg_train_loss = running_loss / len(train_loader_3d)
    
    # 2. طور التقييم الشامل (Comprehensive Validation Phase)
    # نمر الآن على الـ 40 مريضاً بالكامل لحساب متوسط حقيقي مستقر
    val_dices = evaluate_model_3d(model_3d, val_loader_3d, device)
    mean_val_dice = np.mean(val_dices)
    
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]")
    print(f"   -> Train Loss: {avg_train_loss:.4f}")
    print(f"   -> Val Dice   | RV: {val_dices[0]:.4f} | MYO: {val_dices[1]:.4f} | LV: {val_dices[2]:.4f} [Mean: {mean_val_dice:.4f}]")
    
    # 3. إستراتيجية الحفظ الذكي بناءً على الأداء الفعلي للفحص
    if mean_val_dice > best_val_dice:
        best_val_dice = mean_val_dice
        torch.save(model_3d.state_dict(), 'best_mamba_unet_3d.pth')
        print(f"   --> [SAVED] Validation improved! New Best Mean Dice: {best_val_dice:.4f}")
    print("-" * 65)

print("=========================================================================")
print(f"[FINISHED] Training complete. Best Voxel-level Mean Dice: {best_val_dice:.4f}")
print("=========================================================================")

================ STARTING ROBUST 3D TRAINING & VALIDATION ================
Epoch [1/10]
   -> Train Loss: 0.2194
   -> Val Dice   | RV: 0.3335 | MYO: 0.4901 | LV: 0.6177 [Mean: 0.4805]
   --> [SAVED] Validation improved! New Best Mean Dice: 0.4805
-----------------------------------------------------------------
Epoch [2/10]
   -> Train Loss: 0.2075
   -> Val Dice   | RV: 0.4675 | MYO: 0.4979 | LV: 0.6231 [Mean: 0.5295]
   --> [SAVED] Validation improved! New Best Mean Dice: 0.5295
-----------------------------------------------------------------
Epoch [3/10]
   -> Train Loss: 0.1957
   -> Val Dice   | RV: 0.5499 | MYO: 0.5844 | LV: 0.6738 [Mean: 0.6027]
   --> [SAVED] Validation improved! New Best Mean Dice: 0.6027
-----------------------------------------------------------------
Epoch [4/10]
   -> Train Loss: 0.1784
   -> Val Dice   | RV: 0.5586 | MYO: 0.5766 | LV: 0.6837 [Mean: 0.6063]
   --> [SAVED] Validation improved! New Best Mean Dice: 0.6063
-----------------------------------

In [12]:
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

class Down3DV2(nn.Module):
    """Downscaling optimized to preserve vertical Z context for small spatial thickness"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            # الحفاظ على الـ Z slices كما هي وتقليل الارتفاع والعرض فقط
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2)),
            DoubleConv3D(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)

# --- سكربت الاستدلال وحفظ التوقعات الحجمية ثلاثية الأبعاد ---
def run_inference_and_save(model, dataloader, device, save_dir="./predictions"):
    import os
    os.makedirs(save_dir, exist_ok=True)
    
    # تحميل أفضل الأوزان المستقرة
    if os.path.exists('best_mamba_unet_3d.pth'):
        model.load_state_dict(torch.load('best_mamba_unet_3d.pth', map_location=device))
        print("[*] Loaded best weights successfully for inference.")
    
    model.eval()
    print(f"[*] Starting inference on {len(dataloader)} volumes...")
    
    with torch.no_grad():
        for idx, (images, _) in enumerate(dataloader):
            images = images.to(device)
            logits = model(images) # [1, 4, 8, 256, 216]
            
            # تحويل الـ Logits إلى أرقام الفئات عبر Argmax
            preds = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy() # [8, 256, 216]
            
            # حفظ النتائج كملف حجمي طبي H5 للمستودع
            file_path = os.path.join(save_dir, f"patient_pred_{idx:03d}.h5")
            with h5py.File(file_path, 'w') as f:
                f.create_dataset('prediction', data=preds, compression="gzip")
                
    print(f"[SUCCESS] Volumetric predictions saved perfectly inside '{save_dir}/'")

# تفعيل السكربت على بيئة الفحص المستقلة
run_inference_and_save(model_3d, val_loader_3d, device)

[*] Loaded best weights successfully for inference.
[*] Starting inference on 40 volumes...
[SUCCESS] Volumetric predictions saved perfectly inside './predictions/'


In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Down3DVersion2(nn.Module):
    """حفظ السياق العمودي Z وتصغير الارتفاع والعرض فقط"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2)),
            DoubleConv3D(in_channels, out_channels)
        )
    def forward(self, x):
        return self.maxpool_conv(x)

class BiMambaBottleneck(nn.Module):
    """طبقة عنق الزجاجة المطورة للتمرير الحجمي ثنائي الاتجاه"""
    def __init__(self, channels):
        super().__init__()
        self.conv = DoubleConv3D(channels, channels)
        # طبقة إسقاط خطي لدمج التمرير الأمامي والخلفي
        self.merge_project = nn.Linear(channels * 2, channels)

    def forward(self, x):
        b, c, z, h, w = x.shape
        # 1. تسطيح الأبعاد الحجمية إلى متتالية خطية
        flat_seq = x.view(b, c, -1).permute(0, 2, 1) # [Batch, Length, Channels]
        
        # 2. التمرير ثنائي الاتجاه (Forward & Backward Scan Simulation)
        forward_features = flat_seq
        backward_features = torch.flip(flat_seq, dims=[1])
        
        # دمج الميزات المتعاكسة مكانياً لقراءة الحواف بدقة
        bi_features = torch.cat([forward_features, torch.flip(backward_features, dims=[1])], dim=-1)
        merged_seq = self.merge_project(bi_features)
        
        # 3. إعادة التشكيل للأبعاد ثلاثية الأبعاد الأصلية
        mamba_out = merged_seq.permute(0, 2, 1).view(b, c, z, h, w)
        
        # اتصال متبقي (Residual) + تنعيم تلافيفي
        return self.conv(x + mamba_out)

class MambaUNet3DV2(nn.Module):
    def __init__(self, in_channels=1, n_classes=4, base_features=32):
        super().__init__()
        # Encoder مع الحفاظ على الـ Z slices ثابتة عند 8
        self.inc = DoubleConv3D(in_channels, base_features)
        self.down1 = Down3DVersion2(base_features, base_features * 2)
        self.down2 = Down3DVersion2(base_features * 2, base_features * 4)

        # عنق الزجاجة المطور ثنائي الاتجاه
        self.bottleneck = BiMambaBottleneck(base_features * 4)

        # Decoder الصعود الحجمي المتناسق
        self.up1 = Up3D(base_features * 4, base_features * 2)
        self.up2 = Up3D(base_features * 2, base_features)
        self.outc = OutConv3D(base_features, n_classes)

    def forward(self, x):
        x1 = self.inc(x)        # [1, 32, 8, 256, 216]
        x2 = self.down1(x1)    # [1, 64, 8, 128, 108]
        x3 = self.down2(x2)    # [1, 128, 8, 64, 54]  <- لاحظي احتفاظنا بالـ Z=8 كاملة!
        
        x3 = self.bottleneck(x3)
        
        x = self.up1(x3, x2)
        x = self.up2(x, x1)
        return self.outc(x)

# إعادة تهيئة النموذج الجديد وتحديث الـ Optimizer
model_3d = MambaUNet3DV2(in_channels=1, n_classes=4).to(device)
optimizer = torch.optim.AdamW(model_3d.parameters(), lr=2e-4, weight_decay=1e-4)

In [14]:
import numpy as np
import torch

# تمديد التدريب للسماح للمعمارية المحدثة بالوصول لأقصى دقة
NUM_EPOCHS_EXTENDED = 15

print("================ LAUNCHING FULL EXTENDED V2 TRAINING ================")

for epoch in range(NUM_EPOCHS_EXTENDED):
    model_3d.train()
    running_loss = 0.0
    
    for images, labels in train_loader_3d:
        images, labels = images.to(device), labels.to(device).long()
        
        optimizer.zero_grad()
        logits = model_3d(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    avg_train_loss = running_loss / len(train_loader_3d)
    
    # التقييم المستمر
    val_dices = evaluate_model_3d(model_3d, val_loader_3d, device)
    mean_val_dice = np.mean(val_dices)
    
    print(f"Extended Epoch [{epoch+1}/{NUM_EPOCHS_EXTENDED}]")
    print(f"   -> Train Loss: {avg_train_loss:.4f}")
    print(f"   -> Val Dice   | RV: {val_dices[0]:.4f} | MYO: {val_dices[1]:.4f} | LV: {val_dices[2]:.4f} [Mean: {mean_val_dice:.4f}]")
    
    if mean_val_dice > best_val_dice:
        best_val_dice = mean_val_dice
        torch.save(model_3d.state_dict(), 'best_mamba_unet_3d_v2.pth')
        print(f"   --> [SAVED V2] Breakthrough achieved! New Best Mean Dice: {best_val_dice:.4f}")
    print("-" * 65)

print("=========================================================================")
print(f"[COMPLETED] Extended V2 Training. Peak Performance: {best_val_dice:.4f}")
print("=========================================================================")

================ LAUNCHING FULL EXTENDED V2 TRAINING ================
Extended Epoch [1/15]
   -> Train Loss: 1.5757
   -> Val Dice   | RV: 0.0304 | MYO: 0.3169 | LV: 0.4347 [Mean: 0.2606]
-----------------------------------------------------------------
Extended Epoch [2/15]
   -> Train Loss: 1.1734
   -> Val Dice   | RV: 0.0245 | MYO: 0.2947 | LV: 0.4770 [Mean: 0.2654]
-----------------------------------------------------------------
Extended Epoch [3/15]
   -> Train Loss: 0.9198
   -> Val Dice   | RV: 0.1609 | MYO: 0.3823 | LV: 0.4730 [Mean: 0.3388]
-----------------------------------------------------------------
Extended Epoch [4/15]
   -> Train Loss: 0.7283
   -> Val Dice   | RV: 0.4145 | MYO: 0.5370 | LV: 0.7139 [Mean: 0.5551]
-----------------------------------------------------------------
Extended Epoch [5/15]
   -> Train Loss: 0.5695
   -> Val Dice   | RV: 0.4730 | MYO: 0.5598 | LV: 0.7232 [Mean: 0.5853]
-----------------------------------------------------------------
Exten

In [15]:
import numpy as np
import torch

# إعدادات الفحص السريع للمعمارية الجديدة V2
NUM_EPOCHS_V2 = 3
best_val_dice = 0.6180  # نقطة المقارنة السابقة التي نريد كسرها

print("================ STARTING V2 OPTIMIZED 3D TRAINING ================")

for epoch in range(NUM_EPOCHS_V2):
    # 1. طور التدريب (Training Phase)
    model_3d.train()
    running_loss = 0.0
    
    for images, labels in train_loader_3d:
        images, labels = images.to(device), labels.to(device).long()
        
        optimizer.zero_grad()
        logits = model_3d(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    avg_train_loss = running_loss / len(train_loader_3d)
    
    # 2. طور التقييم الشامل (Validation Phase)
    val_dices = evaluate_model_3d(model_3d, val_loader_3d, device)
    mean_val_dice = np.mean(val_dices)
    
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS_V2}]")
    print(f"   -> Train Loss: {avg_train_loss:.4f}")
    print(f"   -> Val Dice   | RV: {val_dices[0]:.4f} | MYO: {val_dices[1]:.4f} | LV: {val_dices[2]:.4f} [Mean: {mean_val_dice:.4f}]")
    
    # 3. حفظ أوزان النموذج المطور في حال كسر الحواجز السابقة
    if mean_val_dice > best_val_dice:
        best_val_dice = mean_val_dice
        torch.save(model_3d.state_dict(), 'best_mamba_unet_3d_v2.pth')
        print(f"   --> [SAVED V2] Architectural improvement confirmed! New Best Mean Dice: {best_val_dice:.4f}")
    print("-" * 65)

print("=========================================================================")
print(f"[FINISHED] V2 Quick Check Done. Highest Mean Dice: {best_val_dice:.4f}")
print("=========================================================================")

================ STARTING V2 OPTIMIZED 3D TRAINING ================
Epoch [1/3]
   -> Train Loss: 0.1630
   -> Val Dice   | RV: 0.7121 | MYO: 0.6640 | LV: 0.8276 [Mean: 0.7346]
   --> [SAVED V2] Architectural improvement confirmed! New Best Mean Dice: 0.7346
-----------------------------------------------------------------
Epoch [2/3]
   -> Train Loss: 0.1591
   -> Val Dice   | RV: 0.6581 | MYO: 0.7312 | LV: 0.8295 [Mean: 0.7396]
   --> [SAVED V2] Architectural improvement confirmed! New Best Mean Dice: 0.7396
-----------------------------------------------------------------
Epoch [3/3]
   -> Train Loss: 0.1506
   -> Val Dice   | RV: 0.7006 | MYO: 0.7457 | LV: 0.8525 [Mean: 0.7663]
   --> [SAVED V2] Architectural improvement confirmed! New Best Mean Dice: 0.7663
-----------------------------------------------------------------
[FINISHED] V2 Quick Check Done. Highest Mean Dice: 0.7663


In [16]:
import numpy as np
import torch
from torch.optim.lr_scheduler import ReduceLROnPlateau

# 1. إعادة تهيئة الأوزان لضمان نظافة البداية
model_3d = MambaUNet3DV2(in_channels=1, n_classes=4).to(device)

# 2. هندسة أدق للمحسن ومعدل التعلم
optimizer = torch.optim.AdamW(model_3d.parameters(), lr=1e-4, weight_decay=1e-4)

# 3. المجدول الذكي المصحح (بدون verbose ليتوافق مع الإصدار الحديث)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

best_val_dice = 0.0  # تصفير العداد لتوثيق الصعود الفعلي الجديد

print("================ LAUNCHING BREAKTHROUGH 3D TRAINING V2.1 ================")

for epoch in range(12):
    model_3d.train()
    running_loss = 0.0
    
    for images, labels in train_loader_3d:
        images, labels = images.to(device), labels.to(device).long()
        
        optimizer.zero_grad()
        logits = model_3d(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    avg_train_loss = running_loss / len(train_loader_3d)
    
    # طور التقييم الشامل على الـ 40 حجماً كاملاً
    val_dices = evaluate_model_3d(model_3d, val_loader_3d, device)
    mean_val_dice = np.mean(val_dices)
    
    # تحديث المجدول بناءً على الـ Mean Dice للتحقق
    scheduler.step(mean_val_dice)
    
    print(f"Epoch [{epoch+1}/12]")
    print(f"   -> Train Loss: {avg_train_loss:.4f}")
    print(f"   -> Val Dice   | RV: {val_dices[0]:.4f} | MYO: {val_dices[1]:.4f} | LV: {val_dices[2]:.4f} [Mean: {mean_val_dice:.4f}]")
    
    if mean_val_dice > best_val_dice:
        best_val_dice = mean_val_dice
        torch.save(model_3d.state_dict(), 'best_mamba_unet_3d_v2_1.pth')
        print(f"   --> [SAVED V2.1] New Peak Performance Confirmed: {best_val_dice:.4f}")
    print("-" * 65)

print("=========================================================================")
print(f"[FINISHED] Training Run Complete. Highest Mean Dice Achieved: {best_val_dice:.4f}")
print("=========================================================================")

================ LAUNCHING BREAKTHROUGH 3D TRAINING V2.1 ================
Epoch [1/12]
   -> Train Loss: 1.7608
   -> Val Dice   | RV: 0.0392 | MYO: 0.0383 | LV: 0.2144 [Mean: 0.0973]
   --> [SAVED V2.1] New Peak Performance Confirmed: 0.0973
-----------------------------------------------------------------
Epoch [2/12]
   -> Train Loss: 1.4600
   -> Val Dice   | RV: 0.0684 | MYO: 0.2971 | LV: 0.5407 [Mean: 0.3021]
   --> [SAVED V2.1] New Peak Performance Confirmed: 0.3021
-----------------------------------------------------------------
Epoch [3/12]
   -> Train Loss: 1.2750
   -> Val Dice   | RV: 0.2729 | MYO: 0.4308 | LV: 0.6617 [Mean: 0.4552]
   --> [SAVED V2.1] New Peak Performance Confirmed: 0.4552
-----------------------------------------------------------------
Epoch [4/12]
   -> Train Loss: 1.1069
   -> Val Dice   | RV: 0.3324 | MYO: 0.5272 | LV: 0.6627 [Mean: 0.5074]
   --> [SAVED V2.1] New Peak Performance Confirmed: 0.5074
----------------------------------------------------

In [17]:
import numpy as np
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau

# 1. إعادة تهيئة النموذج
model_3d = MambaUNet3DV2(in_channels=1, n_classes=4).to(device)

# 2. أوزان الفئات المصممة لدعم الـ RV والـ MYO
class_weights = torch.tensor([0.5, 1.5, 1.5, 1.0]).to(device)
criterion_ce = nn.CrossEntropyLoss(weight=class_weights)

# 3. المحسن والمجدول المشدد
optimizer = torch.optim.AdamW(model_3d.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)

best_val_dice = 0.7413  # الرقم القياسي السابق كهدف للكسر
print("================ LAUNCHING ULTIMATE 3D PUSH V2.2 ================")

for epoch in range(15):
    model_3d.train()
    running_loss = 0.0
    
    for images, labels in train_loader_3d:
        images, labels = images.to(device), labels.to(device).long()
        
        optimizer.zero_grad()
        logits = model_3d(images)
        
        # استخدام criterion المعتمد في بيئتك البرمجية مضافاً إليه وزن الـ CE
        loss = criterion_ce(logits, labels) + criterion(logits, labels)
        
        loss.backward()
        
        # حماية التدرجات ضد التذبذب العنيف
        torch.nn.utils.clip_grad_norm_(model_3d.parameters(), max_norm=1.0)
        
        optimizer.step()
        running_loss += loss.item()
        
    avg_train_loss = running_loss / len(train_loader_3d)
    
    # التقييم
    val_dices = evaluate_model_3d(model_3d, val_loader_3d, device)
    mean_val_dice = np.mean(val_dices)
    
    scheduler.step(mean_val_dice)
    
    print(f"Epoch [{epoch+1}/15] -> Train Loss: {avg_train_loss:.4f}")
    print(f"   -> Val Dice | RV: {val_dices[0]:.4f} | MYO: {val_dices[1]:.4f} | LV: {val_dices[2]:.4f} [Mean: {mean_val_dice:.4f}]")
    
    if mean_val_dice > best_val_dice:
        best_val_dice = mean_val_dice
        torch.save(model_3d.state_dict(), 'ultimate_mamba_unet_3d.pth')
        print(f"   --> 🔥 [BROKEN TARGET] New Record Achieved: {best_val_dice:.4f}")
    print("-" * 65)

print(f"[FINAL REPORT] Highest Dice Reached in V2.2: {best_val_dice:.4f}")

================ LAUNCHING ULTIMATE 3D PUSH V2.2 ================
Epoch [1/15] -> Train Loss: 2.5823
   -> Val Dice | RV: 0.1373 | MYO: 0.2278 | LV: 0.4146 [Mean: 0.2599]
-----------------------------------------------------------------
Epoch [2/15] -> Train Loss: 1.9265
   -> Val Dice | RV: 0.2281 | MYO: 0.3448 | LV: 0.3926 [Mean: 0.3218]
-----------------------------------------------------------------
Epoch [3/15] -> Train Loss: 1.5414
   -> Val Dice | RV: 0.5416 | MYO: 0.5830 | LV: 0.7554 [Mean: 0.6267]
-----------------------------------------------------------------
Epoch [4/15] -> Train Loss: 1.2432
   -> Val Dice | RV: 0.5224 | MYO: 0.5856 | LV: 0.7036 [Mean: 0.6039]
-----------------------------------------------------------------
Epoch [5/15] -> Train Loss: 0.9955
   -> Val Dice | RV: 0.5207 | MYO: 0.5470 | LV: 0.7190 [Mean: 0.5956]
-----------------------------------------------------------------
Epoch [6/15] -> Train Loss: 0.8306
   -> Val Dice | RV: 0.6070 | MYO: 0.6317 | 